In [1]:
import os
import glob
import json
import sys
import time

import pandas as pd
import numpy as np
import rasterio
import rasterstats
import rioxarray as rxr
import geopandas as gpd
import geemap
from osgeo import gdal

In [2]:
import ee
# Trigger the authentication flow.
try:
    ee.Initialize(project='ee-sonle96')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-sonle96')


Successfully saved authorization token.


In [27]:
aez = gpd.read_file(r"C:\Users\sonle\Downloads\WorldCereal_AEZ.geojson")
aez

,fid,aez_id,aez_groupid,tc-annual_eos,tc-annual_sos,tc-wintercereals_sos,tc-wintercereals_eos,tc-maize-main_sos,tc-maize-main_eos,tc-maize-second_sos,tc-maize-second_eos,tc-springcereals_sos,tc-springcereals_eos,geometry
0,1,2008,2000,353,354,149,353,273,154,NaN,NaN,NaN,NaN,"POLYGON ((143.25869 -33.50673, 143.28260 -34.4..."
1,2,2154,2000,347,348,142,347,275,168,360.0,219.0,NaN,NaN,"POLYGON ((-50.89489 -33.52728, -50.89378 -34.4..."
2,3,2161,2000,323,324,143,323,288,136,NaN,NaN,NaN,NaN,"POLYGON ((149.62860 -44.26605, 149.67171 -45.2..."
3,4,3148,3000,349,350,173,349,276,122,NaN,NaN,NaN,NaN,"POLYGON ((137.70427 -35.21140, 137.70482 -35.1..."
4,5,3157,3000,350,351,156,350,290,132,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((178.31709 -41.58726, 178.33675..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,102,48180,48000,249,250,333,119,37,249,320.0,98.0,NaN,NaN,"POLYGON ((86.11651 23.50805, 86.11652 23.50805..."
102,103,49184,49000,264,265,26,193,98,264,NaN,NaN,NaN,NaN,"POLYGON ((131.13283 27.96421, 131.11465 27.018..."
103,104,49186,49000,239,240,22,188,102,239,NaN,NaN,NaN,NaN,"POLYGON ((129.09690 25.22806, 129.09620 24.325..."
104,105,49183,49000,264,265,36,207,105,264,NaN,NaN,NaN,NaN,"POLYGON ((151.85255 45.14807, 151.85259 45.148..."


In [28]:
poly = gpd.read_file(r"D:\Work\ADB\SAR\shapefiles\gadm41_VNM_0.json")
poly.head()

,GID_0,COUNTRY,geometry
0,VNM,Vietnam,"MULTIPOLYGON (((104.87880 8.38180, 104.87910 8..."


In [29]:
aez = aez.to_crs(poly.crs)
intersecting = gpd.sjoin(aez, poly, how="inner", predicate='intersects')
intersecting

,fid,aez_id,aez_groupid,tc-annual_eos,tc-annual_sos,tc-wintercereals_sos,tc-wintercereals_eos,tc-maize-main_sos,tc-maize-main_eos,tc-maize-second_sos,tc-maize-second_eos,tc-springcereals_sos,tc-springcereals_eos,geometry,index_right,GID_0,COUNTRY
39,40,18085,18000,216,217,56,210,110,216,NaN,NaN,NaN,NaN,"POLYGON ((108.72518 8.95096, 108.72505 8.93954...",0,VNM,Vietnam
59,60,26101,26000,200,201,13,186,103,200,NaN,NaN,NaN,NaN,"POLYGON ((110.16336 14.38274, 110.16661 13.478...",0,VNM,Vietnam
60,61,26178,26000,281,282,13,172,80,250,174.0,281.0,NaN,NaN,"POLYGON ((101.86453 19.87338, 101.86457 19.873...",0,VNM,Vietnam
63,64,28108,28000,169,170,351,168,260,169,NaN,NaN,NaN,NaN,"POLYGON ((92.17734 9.86128, 92.17948 8.95737, ...",0,VNM,Vietnam


In [30]:
intersecting.to_file(r"D:\Work\ADB\SAR\shapefiles\wc_aez_vn.geojson", index=False)

In [39]:
ds = rxr.open_rasterio(r"C:\Users\sonle\Downloads\WorldCereal_2021_tc-wintercereals_wintercereals_classification\data\worldcereal_data\MAP-v3\2021\tc-wintercereals\wintercereals\aez\classification\26101_tc-wintercereals_wintercereals_2020-12-29_2021-07-06_classification.tif")


In [40]:
ds

<xarray.DataArray (band: 1, y: 109712, x: 168268)> Size: 18GB
[18461018816 values with dtype=uint8]
Coordinates:
  * band         (band) int32 4B 1
  * x            (x) float64 1MB 100.8 100.8 100.8 100.8 ... 114.9 114.9 114.9
  * y            (y) float64 878kB 20.8 20.8 20.8 20.8 ... 11.66 11.66 11.66
    spatial_ref  int32 4B 0
Attributes: (12/16)
    AEZ_ID:           26101
    AREA_OR_POINT:    Area
    copyright:        ESA WorldCereal project
    creation_time:    2023-04-24 23:01:17
    end_date:         2021-07-06
    legend:           0  other temporary crop\n100  cereals\n254  No crop
    ...               ...
    start_date:       2020-12-29
    title:            ESA-WorldCereal wintercereals map for the 2021 tc-winte...
    _FillValue:       255
    scale_factor:     1.0
    add_offset:       0.0
    long_name:        ESA-WorldCereal wintercereals map for the 2021 tc-winte...

In [3]:
m = gdal.Warp(r"D:\Work\ADB\SAR\world_cereal\winter_crops\vn_winter_cereals.tif",
              glob.glob(r"D:\Work\ADB\SAR\world_cereal\winter_crops\*"),
              cutlineDSName=r"D:\Work\ADB\SAR\shapefiles\gadm41_VNM_0.json",
              cropToCutline=True,
              creationOptions=['COMPRESS=LZW'],
              )
m = None

c:\Users\sonle\anaconda3\envs\geospatial\Lib\site-packages\osgeo\gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [4]:
m = gdal.Warp(r"D:\Work\ADB\SAR\world_cereal\maize\vn_maize.tif",
              glob.glob(r"D:\Work\ADB\SAR\world_cereal\maize\*"),
              cutlineDSName=r"D:\Work\ADB\SAR\shapefiles\gadm41_VNM_0.json",
              cropToCutline=True,
              creationOptions=['COMPRESS=LZW'])

In [45]:
wheat = rxr.open_rasterio(r"D:\Work\ADB\SAR\world_cereal\winter_crops\vn_winter_crops.tif")
wheat

<xarray.DataArray (band: 1, y: 180135, x: 87892)> Size: 16GB
[15832425420 values with dtype=uint8]
Coordinates:
  * band         (band) int32 4B 1
  * x            (x) float64 703kB 102.1 102.1 102.1 102.1 ... 109.5 109.5 109.5
  * y            (y) float64 1MB 23.39 23.39 23.39 23.39 ... 8.382 8.382 8.382
    spatial_ref  int32 4B 0
Attributes: (12/16)
    AEZ_ID:           *
    AREA_OR_POINT:    Area
    copyright:        ESA WorldCereal project
    creation_time:    *
    end_date:         *
    legend:           0  other temporary crop\n100  cereals\n254  No crop
    ...               ...
    start_date:       *
    title:            ESA-WorldCereal wintercereals map for the 2021 tc-winte...
    _FillValue:       255
    scale_factor:     1.0
    add_offset:       0.0
    long_name:        ESA-WorldCereal wintercereals map for the 2021 tc-winte...

In [2]:
from tqdm import tqdm_notebook

paths = glob.glob("D:\Work\ADB\SAR\wheat_south/*")
dfs = []
for p in tqdm_notebook(paths):
    dfs.append(pd.read_csv(p))
df = pd.concat(dfs, axis=0).reset_index(drop=True)
df

<>:3: SyntaxWarning: invalid escape sequence '\W'
<>:3: SyntaxWarning: invalid escape sequence '\W'


C:\Users\sonle\AppData\Local\Temp\ipykernel_28796\3366052013.py:3: SyntaxWarning: invalid escape sequence '\W'
  paths = glob.glob("D:\Work\ADB\SAR\wheat_south/*")
C:\Users\sonle\AppData\Local\Temp\ipykernel_28796\3366052013.py:5: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for p in tqdm_notebook(paths):


  0%|          | 0/792 [00:00<?, ?it/s]

,CropType,Luminosity,NDVI_max,NDVI_mean,NDVI_min,PM25_mean,Precipitation_sum,Rel_humidity_mean,Temperature_max,Temperature_mean,Timestamp,elevation,lat,lon
0,Wheat,0.0,0.517756,0.353236,0.188716,11.338904,0.110568,0.813273,26.826220,25.706761,2017-01-01,2,9.500133,105.161728
1,Wheat,0.0,0.422222,0.272827,0.123431,11.338904,0.110568,0.813273,26.826220,25.706761,2017-01-01,1,9.501032,105.193169
2,Wheat,1.1,0.202432,0.151102,0.099772,11.338904,0.110568,0.813273,26.826220,25.706761,2017-01-01,0,9.501032,105.236288
3,Wheat,0.5,0.310885,0.223066,0.135248,11.338904,0.106291,0.809047,26.873258,25.760616,2017-01-01,2,9.501032,105.266831
4,Wheat,0.0,0.203149,0.174641,0.146133,11.338904,0.110568,0.813273,26.826220,25.706761,2017-01-01,1,9.501930,105.211135
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1551091,Wheat,0.0,0.685001,0.431208,0.103071,31.135925,0.056874,0.768715,27.135126,25.467303,2022-12-01,4,10.499958,106.006144
1551092,Wheat,0.8,0.742539,0.510616,0.100370,31.135925,0.056874,0.768715,27.135126,25.467303,2022-12-01,4,10.499958,106.016026
1551093,Wheat,0.0,0.751838,0.452165,0.090715,31.135925,0.056874,0.768715,27.135126,25.467303,2022-12-01,3,10.499958,106.030399
1551094,Wheat,1.7,0.692137,0.460408,0.182560,31.135925,0.056805,0.775177,27.134719,25.473841,2022-12-01,2,10.499958,106.103162


In [3]:
df.to_csv('datasets/wheat_south.csv', index=False)

In [4]:
from tqdm import tqdm_notebook

paths = glob.glob("D:\Work\ADB\SAR\maize_south/*")
dfs = []
for p in tqdm_notebook(paths):
    dfs.append(pd.read_csv(p))
df = pd.concat(dfs, axis=0).reset_index(drop=True)
df

<>:3: SyntaxWarning: invalid escape sequence '\W'
<>:3: SyntaxWarning: invalid escape sequence '\W'


C:\Users\sonle\AppData\Local\Temp\ipykernel_28796\2333578419.py:3: SyntaxWarning: invalid escape sequence '\W'
  paths = glob.glob("D:\Work\ADB\SAR\maize_south/*")
C:\Users\sonle\AppData\Local\Temp\ipykernel_28796\2333578419.py:5: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for p in tqdm_notebook(paths):


  0%|          | 0/575 [00:00<?, ?it/s]

,CropType,Luminosity,NDVI_max,NDVI_mean,NDVI_min,PM25_mean,Precipitation_sum,Rel_humidity_mean,Temperature_max,Temperature_mean,Timestamp,elevation,lat,lon
0,Maize,0.0,0.521901,0.390091,0.258281,11.338904,0.111213,0.817586,26.865445,25.730831,2017-01-01,2,9.500133,105.145558
1,Maize,0.0,0.293583,0.222229,0.150876,11.338904,0.110568,0.813273,26.826220,25.706761,2017-01-01,1,9.501032,105.212932
2,Maize,0.0,0.174853,0.156378,0.137904,11.338904,0.110568,0.813273,26.826220,25.706761,2017-01-01,4,9.501930,105.152745
3,Maize,0.0,0.234635,0.198255,0.161874,11.338904,0.106291,0.809047,26.873258,25.760616,2017-01-01,5,9.501930,105.290187
4,Maize,0.0,0.504572,0.397296,0.290020,11.338904,0.102025,0.806897,26.876757,25.774882,2017-01-01,2,9.501930,105.382713
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859691,Maize,2.5,0.305921,0.233268,0.077043,28.303875,0.037147,0.792037,27.585158,25.817227,2022-12-01,1,10.490975,106.710423
859692,Maize,22.6,0.280769,0.201466,0.038007,33.537773,0.045452,0.795846,27.410679,25.596220,2022-12-01,0,10.495467,106.537947
859693,Maize,88.1,0.484719,0.261823,0.043298,33.537773,0.045452,0.795846,27.410679,25.596220,2022-12-01,4,10.496365,106.521777
859694,Maize,4.7,0.441753,0.287124,-0.011660,28.303875,0.039319,0.790248,27.489211,25.720642,2022-12-01,4,10.499060,106.639457


In [5]:
df.to_csv('datasets/maize_south.csv', index=False)